# Huấn luyện & Đánh giá Đối chứng: RT-DETR (Vision Transformer)
**Dự án:** Nhận diện 7 Lớp Rác Thải (`trash-ai`)  
**Mục tiêu:** Xây dựng mô hình thực nghiệm đối chứng trường phái Transformer (ngoài họ YOLO) báo cáo trước Hội đồng khoa học.

## 1. Cài đặt Môi trường & Kiểm tra GPU Tesla T4

In [ ]:
!nvidia-smi
!pip install -q ultralytics pandas pyyaml

## 2. Kết nối Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Giải nén Dataset & Thiết lập Cấu hình data_balanced.yaml (Train, Val, Test)

In [ ]:
import os, zipfile

zip_candidates = [
    '/content/drive/MyDrive/Trash (3).zip',
    '/content/drive/MyDrive/trash (3).zip',
    '/content/Trash (3).zip'
]
zip_file = next((z for z in zip_candidates if os.path.exists(z)), None)
extract_path = '/content/trash_project'

if zip_file and not os.path.exists(os.path.join(extract_path, 'Trash_dataset_balanced')):
    print(f"📦 Đang giải nén {zip_file} vào SSD Colab...")
    with zipfile.ZipFile(zip_file, 'r') as z:
        z.extractall(extract_path)

dataset_dir = None
for root, dirs, files in os.walk(extract_path):
    if os.path.basename(root) == 'Trash_dataset_balanced':
        dataset_dir = root
        break

assert dataset_dir is not None and os.path.exists(dataset_dir), '❌ Không tìm thấy thư mục Trash_dataset_balanced!'

test_img_dir = os.path.join(dataset_dir, 'test', 'images')
has_test = os.path.exists(test_img_dir)

yaml_content = f"""path: {os.path.abspath(dataset_dir)}
train: train/images
val: val/images
test: {'test/images' if has_test else 'val/images'}

nc: 7
names:
  0: battery
  1: cardboard
  2: paper
  3: glass
  4: metal
  5: plastic
  6: organic
"""
with open('/content/data_balanced.yaml', 'w', encoding='utf-8') as f:
    f.write(yaml_content.strip())
print('✅ Cấu hình /content/data_balanced.yaml đã sẵn sàng (đầy đủ train, val, test)!')

## 4. Huấn luyện RT-DETR (Real-Time Detection Transformer)
- Optimizer: `AdamW`, Learning rate: `1e-4` chuẩn Transformer.
- Tự động đồng bộ checkpoint (`best.pt`, `results.csv`) về Google Drive sau mỗi Epoch.

In [ ]:
import os, shutil, torch
from ultralytics import RTDETR

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"⚡ GPU: {torch.cuda.get_device_name(0)} (CUDA Active)")

local_project = '/content/runs/detect'
exp_name = 'trash_rtdetr_evidence'
local_save_dir = f'{local_project}/{exp_name}'
drive_save_dir = f'/content/drive/MyDrive/Trash_Runs/{exp_name}'

os.makedirs(drive_save_dir, exist_ok=True)
os.makedirs(f'{drive_save_dir}/weights', exist_ok=True)

# Đồng bộ an toàn từng file
def safe_sync(src, dst):
    if os.path.exists(src):
        try:
            if os.path.exists(dst):
                os.remove(dst)
            shutil.copy2(src, dst)
        except Exception:
            pass

def on_fit_epoch_end(trainer):
    for fname in ['results.csv', 'args.yaml']:
        safe_sync(f'{local_save_dir}/{fname}', f'{drive_save_dir}/{fname}')
    for wname in ['last.pt', 'best.pt']:
        safe_sync(f'{local_save_dir}/weights/{wname}', f'{drive_save_dir}/weights/{wname}')

# Nạp mô hình RT-DETR Large pretrained
model = RTDETR('rtdetr-l.pt')
model.add_callback('on_fit_epoch_end', on_fit_epoch_end)

print("🚀 BẮT ĐẦU HUẤN LUYỆN RT-DETR (100 EPOCHS, BATCH=16, ADAMW)...")
model.train(
    data='/content/data_balanced.yaml',
    epochs=100,
    batch=16,
    imgsz=640,
    device=0,
    workers=2,
    optimizer='AdamW',
    lr0=0.0001,
    patience=10,
    cos_lr=False,
    weight_decay=0.0005,
    amp=True,
    project=local_project,
    name=exp_name,
    exist_ok=True,
    save=True,
    verbose=True
)

# Sao chép toàn bộ biểu đồ, ma trận nhầm lẫn sang Google Drive
if os.path.exists(local_save_dir):
    shutil.copytree(local_save_dir, drive_save_dir, dirs_exist_ok=True)
    print(f'✅ Đã sao lưu toàn bộ artifacts và biểu đồ sang Google Drive: {drive_save_dir}')

## 5. Đánh giá Độc lập trên tập Test Set (`split='test'`) & Xuất Bằng chứng Thực nghiệm

In [ ]:
import os, shutil
import pandas as pd
from ultralytics import RTDETR

exp_name = 'trash_rtdetr_evidence'
best_candidates = [
    f'/content/runs/detect/{exp_name}/weights/best.pt',
    f'/content/drive/MyDrive/Trash_Runs/{exp_name}/weights/best.pt'
]
best_pt = next((b for b in best_candidates if os.path.exists(b)), None)
assert best_pt is not None, '❌ Không tìm thấy weights best.pt!'

print(f'🏆 Nạp trọng số tốt nhất: {best_pt}')
eval_model = RTDETR(best_pt)

print('📊 Đang chạy kiểm thử trên tập Test Set độc lập...')
test_res = eval_model.val(
    data='/content/data_balanced.yaml',
    split='test',
    imgsz=640,
    device=0,
    verbose=True
)

param_count = round(sum(p.numel() for p in eval_model.model.parameters()) / 1e6, 2)
latency_ms = test_res.speed.get('inference', 0.0)
fps = (1000.0 / latency_ms) if latency_ms > 0 else 0.0

p_overall = float(test_res.box.mp)
r_overall = float(test_res.box.mr)
map50_overall = float(test_res.box.map50)
map_overall = float(test_res.box.map)

report_lines = []
report_lines.append('=' * 75)
report_lines.append('           BÁO CÁO THỰC NGHIỆM ĐỐI CHỨNG RT-DETR TRÊN TẬP TEST SET')
report_lines.append('=' * 75)
report_lines.append(f'Kiến trúc           : RT-DETR (Real-Time Detection Transformer)')
report_lines.append(f'Số lượng tham số    : {param_count} M (Triệu parameters)')
report_lines.append(f'Độ trễ suy luận     : {latency_ms:.2f} ms (~{fps:.1f} FPS trên Tesla T4)')
report_lines.append('-' * 75)
report_lines.append(f'Precision tổng thể  : {p_overall:.4f} ({p_overall*100:.2f}%)')
report_lines.append(f'Recall tổng thể     : {r_overall:.4f} ({r_overall*100:.2f}%)')
report_lines.append(f'mAP@50 tổng thể     : {map50_overall:.4f} ({map50_overall*100:.2f}%)')
report_lines.append(f'mAP@50-95 tổng thể  : {map_overall:.4f} ({map_overall*100:.2f}%)')
report_lines.append('-' * 75)
report_lines.append(f"{'Lớp (Class)':<14} | {'Precision':<10} | {'Recall':<10} | {'mAP@50':<10} | {'mAP@50-95':<10}")
report_lines.append('-' * 75)

class_names = list(test_res.names.values())
per_class_records = []

for i, cname in enumerate(class_names):
    try:
        p_cls, r_cls, map50_cls, map_cls = test_res.box.class_result(i)
    except Exception:
        p_cls = r_cls = 0.0
        map50_cls = float(test_res.box.map50)
        map_cls = float(test_res.box.maps[i]) if i < len(test_res.box.maps) else 0.0
    
    line = f"{cname:<14} | {p_cls:<10.4f} | {r_cls:<10.4f} | {map50_cls:<10.4f} | {map_cls:<10.4f}"
    report_lines.append(line)
    per_class_records.append({
        'class': cname,
        'precision': round(float(p_cls), 4),
        'recall': round(float(r_cls), 4),
        'map50': round(float(map50_cls), 4),
        'map50_95': round(float(map_cls), 4)
    })

report_lines.append('=' * 75)
report_text = '\n'.join(report_lines)
print('\n' + report_text)

local_dir = f'/content/runs/detect/{exp_name}'
drive_dir = f'/content/drive/MyDrive/Trash_Runs/{exp_name}'

report_txt_path = f'{local_dir}/test_evaluation_report.txt'
with open(report_txt_path, 'w', encoding='utf-8') as f:
    f.write(report_text)

df_classes = pd.DataFrame(per_class_records)
csv_report_path = f'{local_dir}/test_per_class_metrics.csv'
df_classes.to_csv(csv_report_path, index=False)

if os.path.exists(drive_dir):
    shutil.copy2(report_txt_path, f'{drive_dir}/test_evaluation_report.txt')
    shutil.copy2(csv_report_path, f'{drive_dir}/test_per_class_metrics.csv')
    print(f'✅ Đã lưu bằng chứng đối chứng (Report & CSV) sang Google Drive: {drive_dir}')